In [1]:
import torch
from transformers import AutoTokenizer

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [2]:
# -- Manual tokenizer

def tokenize(text):
    return text.lower().split()

def build_vocab(sentences):
    vocab = {}
    for sent in sentences:
        for token in tokenize(sent):
            if token not in vocab:
                vocab[token] = len(vocab) + 1 # 0 preserved for padding
    return vocab

sentences = ["I love my dog", "I love my cat"]
vocab = build_vocab(sentences)

print(f"Manual tokenizer: {vocab}")            

Manual tokenizer: {'i': 1, 'love': 2, 'my': 3, 'dog': 4, 'cat': 5}


In [3]:
# Problem: "puppy" is not in vocab → Out-of-Vocabulary (OOV)!
new_text = "I love my puppy"
tokens = tokenize(new_text)
ids = [vocab.get(t, 0) for t in tokens]  # 0 = unknown
print(f"OOV example: {tokens} → {ids}")  # [..., 0] — lost information!

OOV example: ['i', 'love', 'my', 'puppy'] → [1, 2, 3, 0]


In [4]:
# --- HuggingFace tokenizer: solves OOV with subword tokenization ---
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# BERT uses WordPiece: unknown words get split into known subwords
encoded_inputs = tokenizer(sentences, padding=True, return_tensors="pt")
print(f"\nBERT tokenized:")
print(f"  input_ids: {encoded_inputs['input_ids']}")
print(f"  tokens: {[tokenizer.convert_ids_to_tokens(ids) for ids in encoded_inputs['input_ids']]}")

# Subword example: "tokenization" → ["token", "##ization"]
subword_result = tokenizer("tokenization is important")
print(f"\nSubword split: {tokenizer.convert_ids_to_tokens(subword_result['input_ids'])}")
# ['[CLS]', 'token', '##ization', 'is', 'important', '[SEP]']

# Special tokens
print(f"\nSpecial tokens: CLS={tokenizer.cls_token_id}  SEP={tokenizer.sep_token_id}")
print(f"Vocab size: {tokenizer.vocab_size}")  # 30522


BERT tokenized:
  input_ids: tensor([[ 101, 1045, 2293, 2026, 3899,  102],
        [ 101, 1045, 2293, 2026, 4937,  102]])
  tokens: [['[CLS]', 'i', 'love', 'my', 'dog', '[SEP]'], ['[CLS]', 'i', 'love', 'my', 'cat', '[SEP]']]

Subword split: ['[CLS]', 'token', '##ization', 'is', 'important', '[SEP]']

Special tokens: CLS=101  SEP=102
Vocab size: 30522
